Формат задания

Выполните задание локально на своём устройстве и проверьте результаты самостоятельно: нажмите кнопку «Готово» и сверьте свой вариант с авторским решением.  
Что нужно сделать

В этом задании вы реализуете метод backward(y_true) в классе MLP. Метод должен пройти от выходного слоя к входному и вычислить градиенты весов и смещений для каждого слоя по формулам backpropagation. Для выходного слоя используется линейная активация и MSE: дельта на выходе равна 2*(aL - y_true)/m. Сохраните градиенты в списках self.dW_list и self.db_list в том же порядке, что и self.W и self.b (т. е. dW_list[i] соответствует W[i]).

In [ ]:
class MLP:
    def __init__(self, layer_sizes, activation="sigmoid"):
        self.layer_sizes = layer_sizes
        self.activation = activation
        self.W, self.b = [], []
        # сюда будем складывать Z и A, а также градиенты
        self.Z_list, self.A_list = [], []
        self.dW_list, self.db_list = [], []

        for i in range(len(layer_sizes) - 1):
            in_dim, out_dim = layer_sizes[i], layer_sizes[i + 1]
            self.W.append(np.random.randn(in_dim, out_dim) * 0.1)
            self.b.append(np.zeros((1, out_dim)))

    def _sigmoid(self, Z):
        return 1 / (1 + np.exp(-Z))

    def _dactivate(self, Z):
        # Производная сигмоида через Z
        s = self._sigmoid(Z)
        return s * (1 - s)

    def mse(self, y_pred, y_true):
        return np.mean((y_pred - y_true) ** 2)

    def forward(self, X):
        self.Z_list, self.A_list = [], []
        A = X
        self.A_list.append(A)

        for i in range(len(self.W)):
            Z = A.dot(self.W[i]) + self.b[i]
            self.Z_list.append(Z)

            # скрытые слои – sigmoid
            if i < len(self.W) - 1:
                A = self._sigmoid(Z)
            else:
                # выходной слой – линейный
                A = Z

            self.A_list.append(A)

        return A

    def backward(self, y_true):
        m = y_true.shape[0]

        # δ на выходном слое (линейный + MSE)
        aL = self.A_list[-1]
        delta = 2 * (aL - y_true) / m  # δ^L = 2*(A^L - Y)/m

        # Градиенты для выходного слоя
        a_prev = self.A_list[-2]
        dW = a_prev.T.dot(delta)  # dW^L = A^{L-1}ᵀ · δ^L
        db = np.sum(delta, axis=0, keepdims=True)
        self.dW_list, self.db_list = [dW], [db]

        # 3) Обратный проход по скрытым слоям (sigmoid)
        for l in range(len(self.layer_sizes) - 2, 0, -1):
            z = self.Z_list[l - 1]
            a_prev = self.A_list[l - 1]
            W_next = self.W[l]

            # δ^l = (δ^{l+1} · W^{l+1}ᵀ) * σ'(z^l)
            delta = delta.dot(W_next.T) * self._dactivate(z)

            dW = a_prev.T.dot(delta)  # dW^l = A^{l-1}ᵀ · δ^l
            db = np.sum(delta, axis=0, keepdims=True)

            self.dW_list.insert(0, dW)  # вставляем вперёд
            self.db_list.insert(0, db)